[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/primers/primer-3-numpy.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Primer 3 — NumPy

**MSIS · AME 5003 and AME 5053 · Practice notebook · about 1 hour · not assessed**

*Arrays, axes, broadcasting and the dot product — the operations every document vector, weight
matrix and similarity score in this course is made of.*

From Unit II onwards, text has already been turned into numbers, and everything after that is
arithmetic on rectangles of them. A term-document matrix is a rectangle. A batch of TF-IDF
vectors is a rectangle. A layer of a neural network is a rectangle multiplied by another
rectangle. NumPy is the library that holds them and does the arithmetic, and scikit-learn,
gensim and PyTorch all hand you NumPy arrays or something modelled closely on them.

This notebook is practice, not assessment. Everything in it is used from lab 4 onwards, and the
small matrix it works with throughout has the orientation session 8 uses for the incidence
matrix: rows are terms, columns are documents.

**By the end you will be able to:**

1. Make arrays, and read `shape` and `dtype` before anything else
2. Index, slice and mask, and know when you are holding a view rather than a copy
3. Sum and average along an axis, and say which axis you want and why
4. Use broadcasting deliberately, and read the error when it fails
5. Compute cosine similarity for a whole matrix at once, and rank the results

---
## Part 0 — Setup

NumPy arrives with Colab, so there is nothing to install. The import line is a convention rather
than a rule, but it is a universal one: every piece of NumPy code you will ever read says `np`.

In [ ]:
import numpy as np

print(np.__version__)
# Verified output:
#   2.5.1

---
## Part 1 — Arrays, and why not a list

A Python list can hold anything, in any mix, and each element is a separate object somewhere in
memory. A NumPy array holds one type, in one contiguous block, with a known shape. That
restriction is what buys the speed, and it is why an array knows things a list does not.

In [ ]:
counts = np.array([3, 0, 1, 7, 2])

print("array:  ", counts)
print("shape:  ", counts.shape)     # a tuple, one entry per dimension
print("dtype:  ", counts.dtype)     # the one type every element has
print("ndim:   ", counts.ndim)      # how many dimensions
print("size:   ", counts.size)      # how many elements in total
# Verified output:
#   array:   [3 0 1 7 2]
#   shape:   (5,)
#   dtype:   int64
#   ndim:    1
#   size:    5

**Print `shape` first when something goes wrong.** Most NumPy errors in this course are shape
errors, and most of them are visible the moment you look. `dtype` is the second thing to check:
an array of integers divides differently from an array of floats, and an array that has silently
become `dtype=object` has lost all of the speed.

A two-dimensional array is a list of rows. This one is a **term-document matrix** with three
terms and four documents — the shape all of Unit II is built on.

In [ ]:
matrix = np.array([
    [1, 0, 2, 1],
    [0, 3, 0, 1],
    [4, 1, 1, 0],
])

print(matrix)
print()
print("shape:", matrix.shape, " -> ", matrix.shape[0], "rows,", matrix.shape[1], "columns")
# Verified output:
#   [[1 0 2 1]
#    [0 3 0 1]
#    [4 1 1 0]]
#
#   shape: (3, 4)  ->  3 rows, 4 columns

Read `(3, 4)` as "three rows of four", and remember that rows come first in every shape tuple you
will see. In this matrix a row is a term and a column is a document, so the entry at row 2,
column 1 is the number of times term 2 occurs in document 1.

### Making arrays without typing them

You rarely type an array out. These are the constructors worth knowing by name.

In [ ]:
print("zeros:   ", np.zeros(4))
print("ones 2x3:\n", np.ones((2, 3)))
print("arange:  ", np.arange(0, 10, 2))      # start, stop (excluded), step
print("linspace:", np.linspace(0, 1, 5))     # start, stop (included), how many
print("eye:\n", np.eye(3))                   # the identity matrix
# Verified output:
#   zeros:    [0. 0. 0. 0.]
#   ones 2x3:
#    [[1. 1. 1.]
#    [1. 1. 1.]]
#   arange:   [0 2 4 6 8]
#   linspace: [0.   0.25 0.5  0.75 1.  ]
#   eye:
#    [[1. 0. 0.]
#    [0. 1. 0.]
#    [0. 0. 1.]]

In [ ]:
rng = np.random.default_rng(seed=42)

print(rng.random((2, 3)))
print()
print(rng.integers(0, 10, size=5))
# Verified output:
#   [[0.77395605 0.43887844 0.85859792]
#    [0.69736803 0.09417735 0.97562235]]
#
#   [7 7 7 7 5]

`default_rng(seed=...)` is the modern way to get random numbers, and the seed is what makes a
result reproducible: the same seed gives the same numbers on any machine. Set one in every
notebook that initialises weights or samples data, or you cannot compare two runs — and you will
want to, from session 20 onwards.

### Why it is faster

The speed is not a detail. It is the reason the library exists.

In [ ]:
import time

n = 1_000_000
py_list = list(range(n))
np_array = np.arange(n)

start = time.perf_counter()
[x * 2 for x in py_list]
list_time = time.perf_counter() - start

start = time.perf_counter()
np_array * 2
array_time = time.perf_counter() - start

print(f"  list comprehension {list_time * 1000:7.1f} ms")
print(f"  numpy              {array_time * 1000:7.1f} ms")
print(f"  {list_time / array_time:.0f}x")
# Verified output:
#     list comprehension    22.9 ms
#     numpy                  0.9 ms
#     26x

`np_array * 2` has no Python loop in it. The multiplication happens inside compiled code, over a
block of memory, one type. Writing `for` over an array is the commonest way to lose that, and the
usual fix is to look for the operation NumPy already has — which is what the rest of this
notebook is about.

---
## Part 2 — Indexing and slicing

Indexing a 2-D array takes a row and a column together, separated by a comma, in that order.
People coming from lists write `matrix[1][2]`, which works but builds an intermediate row;
`matrix[1, 2]` is the NumPy way.

In [ ]:
print("matrix[1, 2]  ", matrix[1, 2], "  one element: row 1, column 2")
print("matrix[1]     ", matrix[1], "  a whole row  -> term 1 across all documents")
print("matrix[:, 2]  ", matrix[:, 2], "  a whole column -> document 2 across all terms")
print("matrix[:2, 1:]\n", matrix[:2, 1:])
# Verified output:
#   matrix[1, 2]   0   one element: row 1, column 2
#   matrix[1]      [0 3 0 1]   a whole row  -> term 1 across all documents
#   matrix[:, 2]   [2 0 1]   a whole column -> document 2 across all terms
#   matrix[:2, 1:]
#    [[0 2 1]
#    [3 0 1]]

The colon means "everything along this axis". `matrix[:, 2]` is the vector for document 2, and
`matrix[1]` is the vector for term 1 — the two views of the same matrix that session 8 keeps
switching between.

Negative indices count from the end, and slices are half-open, exactly as in Python lists:
`matrix[:2]` gives rows 0 and 1.

### The trap: a slice is a view, not a copy

In [ ]:
original = np.array([10, 20, 30, 40, 50])

part = original[1:4]
part[0] = 999

print("part:    ", part)
print("original:", original, "  <- changed, and nothing said so")
# Verified output:
#   part:     [999  30  40]
#   original: [ 10 999  30  40  50]   <- changed, and nothing said so

A slice of a NumPy array shares memory with the array it came from, so writing to the slice
writes to the original. This is deliberate — it is what makes slicing free, even on a large
matrix — but it is not what a Python list does, and it produces bugs that look like corrupted
data rather than like an aliasing mistake.

When you want an independent array, ask for one with `.copy()`.

In [ ]:
original = np.array([10, 20, 30, 40, 50])

part = original[1:4].copy()
part[0] = 999

print("part:    ", part)
print("original:", original, "  <- untouched")
# Verified output:
#   part:     [999  30  40]
#   original: [10 20 30 40 50]   <- untouched

### Exercise 1

From `matrix` above, print: the vector for document 3, the vector for term 0, the submatrix of
the first two terms and last two documents, and the total number of term occurrences in the
whole matrix.

In [ ]:
# YOUR CODE HERE
# document 3      -> a column
# term 0          -> a row
# first two terms, last two documents -> a slice on both axes
# total           -> matrix.sum()

---
## Part 3 — Boolean masks

Comparing an array with a number gives an array of booleans, one per element. Indexing with that
mask keeps the elements where it is `True`. This is how filtering is written in NumPy, and once
it is familiar you will stop writing loops for it.

In [ ]:
counts = np.array([3, 0, 1, 7, 2, 0, 5])

mask = counts > 2

print("counts:", counts)
print("mask:  ", mask)
print("kept:  ", counts[mask])
print("how many:", mask.sum(), " <- True counts as 1")
# Verified output:
#   counts: [3 0 1 7 2 0 5]
#   mask:   [ True False False  True False False  True]
#   kept:   [3 7 5]
#   how many: 3  <- True counts as 1

`mask.sum()` counting the `True` values is an idiom worth recognising: it is how you answer "in
how many documents does this term appear?", which is the document frequency in TF-IDF.

Combine conditions with `&` and `|`, and put brackets round each one — `and` and `or` do not work
on arrays, and the error message when you forget is memorably unhelpful.

In [ ]:
print("counts between 1 and 5:", counts[(counts > 1) & (counts < 5)])
print()
print("where are the zeros:", np.where(counts == 0)[0])   # the positions
print("zeros replaced by -1:", np.where(counts == 0, -1, counts))
# Verified output:
#   counts between 1 and 5: [3 2]
#
#   where are the zeros: [1 5]
#   zeros replaced by -1: [ 3 -1  1  7  2 -1  5]

`np.where` has two uses that look unrelated. With one argument it returns the positions where the
condition holds; with three, it chooses element by element between two alternatives. The second
form is how you avoid a division by zero in a smoothing formula without writing a loop, which is
session 13's arithmetic.

---
## Part 4 — Axes: the rule worth memorising

`sum`, `mean`, `max` and their relatives take an `axis` argument, and getting it the wrong way
round is the commonest NumPy mistake in this course. The rule is short:

**The axis you name is the axis that disappears.**

Our matrix is terms × documents, shape `(3, 4)`. Summing along axis 0 collapses the three terms
and leaves one number per document; summing along axis 1 collapses the four documents and leaves
one number per term.

In [ ]:
print("matrix:\n", matrix)
print()
print("matrix.sum()        ", matrix.sum(), "        everything, one number")
print("matrix.sum(axis=0)  ", matrix.sum(axis=0), "  shape", matrix.sum(axis=0).shape,
      " one per document")
print("matrix.sum(axis=1)  ", matrix.sum(axis=1), "     shape", matrix.sum(axis=1).shape,
      " one per term")
# Verified output:
#   matrix:
#    [[1 0 2 1]
#    [0 3 0 1]
#    [4 1 1 0]]
#
#   matrix.sum()         14         everything, one number
#   matrix.sum(axis=0)   [5 4 3 2]   shape (4,)  one per document
#   matrix.sum(axis=1)   [4 4 6]      shape (3,)  one per term

Check the shapes rather than the numbers. Axis 0 had length 3 and the result has length 4; axis 1
had length 4 and the result has length 3. Whichever axis you name is gone from the shape, and
that check settles the question every time without any thinking about rows and columns.

Two of these sums have names you already know. The per-document sum is the length of each
document in tokens. The per-term sum is the collection frequency of each term.

In [ ]:
print("document lengths:  ", matrix.sum(axis=0))
print("collection freqs:  ", matrix.sum(axis=1))
print("document frequency:", (matrix > 0).sum(axis=1), " <- documents each term appears in")
# Verified output:
#   document lengths:   [5 4 3 2]
#   collection freqs:   [4 4 6]
#   document frequency: [3 2 3]  <- documents each term appears in

That last line is the document frequency of TF-IDF, in one expression: turn the counts into
booleans, then count the `True`s along the document axis. Session 10 defines it; this is how it
is computed.

`keepdims=True` is worth knowing now, because part 5 needs it. It keeps the collapsed axis in the
shape, with length 1, which is what makes the result line up for broadcasting.

In [ ]:
print("without keepdims:", matrix.sum(axis=0).shape)
print("with keepdims:   ", matrix.sum(axis=0, keepdims=True).shape)
# Verified output:
#   without keepdims: (4,)
#   with keepdims:    (1, 4)

### Exercise 2

Using `matrix`, compute and print:

1. The average number of occurrences per document (one number per document)
2. Which document is the longest, using `argmax`
3. Which terms appear in at least three of the four documents

In [ ]:
# YOUR CODE HERE
# 1. matrix.mean(axis=?)
# 2. np.argmax(...) gives the position of the largest value
# 3. a mask, a sum along an axis, and np.where(... >= 3)

---
## Part 5 — Broadcasting

Arithmetic between arrays of different shapes works when NumPy can stretch the smaller one to fit
the larger, without copying it. That stretching is **broadcasting**, and it is why so little
NumPy code has loops in it.

The rule is applied to the shapes, right to left. Two axes are compatible when they are equal, or
when one of them is 1.

In [ ]:
print("matrix + 10:\n", matrix + 10)      # scalar: stretched over everything
print()

per_document = np.array([1, 2, 3, 4])     # shape (4,) matches the last axis
print("matrix * per_document:\n", matrix * per_document)
# Verified output:
#   matrix + 10:
#    [[11 10 12 11]
#    [10 13 10 11]
#    [14 11 11 10]]
#
#   matrix * per_document:
#    [[1 0 6 4]
#    [0 6 0 4]
#    [4 2 3 0]]

The second one multiplied each column by its own number: shape `(3, 4)` against shape `(4,)`
lines up on the last axis, and the array of four is reused for every row.

To scale each **row** instead, the shape has to be `(3, 1)` — which is exactly what `keepdims`
gives you.

In [ ]:
per_term = np.array([[10], [100], [1000]])       # shape (3, 1)

print("shapes:", matrix.shape, "and", per_term.shape)
print(matrix * per_term)
# Verified output:
#   shapes: (3, 4) and (3, 1)
#   [[  10    0   20   10]
#    [   0  300    0  100]
#    [4000 1000 1000    0]]

When the shapes cannot be lined up, NumPy raises `ValueError` and tells you both of them. This is
the most useful error message in the library, and it is worth seeing on purpose once.

In [ ]:
try:
    matrix * np.array([1, 2, 3])       # (3, 4) against (3,) — the 3 lands on the wrong axis
except ValueError as e:
    print("ValueError:", e)
# Verified output:
#   ValueError: operands could not be broadcast together with shapes (3,4) (3,)

Read it as "the last axes disagree: 4 against 3". The fix is almost always to reshape the smaller
array to `(3, 1)`, so that its 3 lines up with the matrix's rows rather than its columns.

---
## Part 6 — The dot product, and cosine similarity

The dot product multiplies two vectors element by element and adds the results. In NumPy the `@`
operator does it, for vectors and for matrices alike; `np.dot` is the older spelling of the same
thing.

In [ ]:
a = np.array([1, 0, 2])
b = np.array([2, 1, 1])

print("a @ b:", a @ b, " =", 1 * 2, "+", 0 * 1, "+", 2 * 1)
# Verified output:
#   a @ b: 4  = 2 + 0 + 2

Two documents sharing many terms give a large dot product, which is why it is the starting point
for similarity. On its own it is unusable, because a long document scores highly against
everything simply for being long. Dividing by both lengths removes that, and the result is the
**cosine similarity** of session 19:

    cos(a, b) = (a @ b) / (norm(a) * norm(b))

Written for one pair, that is one line.

In [ ]:
d0 = matrix[:, 0].astype(float)      # document 0 as a vector over terms
d3 = matrix[:, 3].astype(float)

cosine = (d0 @ d3) / (np.linalg.norm(d0) * np.linalg.norm(d3))

print("d0:", d0)
print("d3:", d3)
print("cosine:", round(float(cosine), 3))
# Verified output:
#   d0: [1. 0. 4.]
#   d3: [1. 1. 0.]
#   cosine: 0.171

`np.linalg.norm` is the Euclidean length of the vector — the square root of the sum of the
squares. Note the `.astype(float)`: our matrix holds integers, and integer arithmetic would floor
the division. Converting once, early, avoids a whole class of quiet mistakes.

### Every pair at once

You almost never want one pair. Normalise every document vector to unit length, and then a single
matrix multiplication gives every cosine at once.

In [ ]:
docs = matrix.T.astype(float)                       # rows are now documents
print("docs shape:", docs.shape, " (4 documents, 3 terms)")

lengths = np.linalg.norm(docs, axis=1, keepdims=True)   # one length per row
unit = docs / lengths                                    # broadcasting does the division

similarity = unit @ unit.T                               # (4,3) @ (3,4) -> (4,4)

print()
print(np.round(similarity, 3))
# Verified output:
#   docs shape: (4, 3)  (4 documents, 3 terms)
#
#   [[1.    0.307 0.651 0.171]
#    [0.307 1.    0.141 0.671]
#    [0.651 0.141 1.    0.632]
#    [0.171 0.671 0.632 1.   ]]

Three lines, and every one of them is something from earlier in this notebook: `axis` with
`keepdims`, broadcasting, and a matrix product. That is the whole of lab 7's similarity step, and
scikit-learn's `cosine_similarity` does exactly this.

Read the result as a table. The diagonal is 1, because every document is identical to itself. The
matrix is symmetric, because cosine does not care which document you ask about first. And the
largest off-diagonal entry names the most similar pair.

### Exercise 3

Using `similarity`, find the document most similar to document 0, excluding document 0 itself.

`argsort` sorts and returns positions rather than values, in ascending order, so the last entry
is the largest and `[::-1]` reverses it.

In [ ]:
# YOUR CODE HERE
# row = similarity[0]
# order = np.argsort(row)[::-1]
# the first entry of order is document 0 itself — skip it

That loop is a search engine's result list. A query is one more vector, the scores are one more
matrix product, and the ranking is this `argsort` — which is why session 10 can say that ranked
retrieval is TF-IDF plus a sort, and mean it literally.

---
## Part 7 — Reshaping, and the shapes of Unit III

From session 20 onwards you will meet shape errors constantly, because a neural network is a
sequence of matrix multiplications and each one has an opinion about its inputs. Three operations
account for most of the fixing.

In [ ]:
v = np.arange(12)

print("v:      ", v, v.shape)
print("reshape:\n", v.reshape(3, 4))
print()
print("reshape(-1, 2) — the -1 means 'work it out':\n", v.reshape(-1, 2).shape)
# Verified output:
#   v:       [ 0  1  2  3  4  5  6  7  8  9 10 11] (12,)
#   reshape:
#    [[ 0  1  2  3]
#    [ 4  5  6  7]
#    [ 8  9 10 11]]
#
#   reshape(-1, 2) — the -1 means 'work it out':
#    (6, 2)

In [ ]:
row = np.array([1, 2, 3])

print("row.shape:            ", row.shape)
print("row[np.newaxis, :]:   ", row[np.newaxis, :].shape, " a 1x3 row")
print("row[:, np.newaxis]:   ", row[:, np.newaxis].shape, " a 3x1 column")
print("transpose of the matrix:", matrix.shape, "->", matrix.T.shape)
# Verified output:
#   row.shape:             (3,)
#   row[np.newaxis, :]:    (1, 3)  a 1x3 row
#   row[:, np.newaxis]:    (3, 1)  a 3x1 column
#   transpose of the matrix: (3, 4) -> (4, 3)

`reshape` rearranges the same elements into a new shape, and `-1` in one position means "compute
this one from the total". `np.newaxis` inserts an axis of length 1, which is how you turn a
vector into a one-row or one-column matrix — and, in Unit III, how you add the batch dimension a
model expects. `.T` transposes.

Stacking is the fourth: `np.vstack` puts arrays on top of each other, `np.hstack` side by side.
Building a document-term matrix out of a list of per-document vectors is a `vstack`.

In [ ]:
v1 = np.array([1, 0, 2])
v2 = np.array([0, 3, 0])

print("vstack:\n", np.vstack([v1, v2]), np.vstack([v1, v2]).shape)
print("hstack:", np.hstack([v1, v2]), np.hstack([v1, v2]).shape)
# Verified output:
#   vstack:
#    [[1 0 2]
#    [0 3 0]] (2, 3)
#   hstack: [1 0 2 0 3 0] (6,)

---
## Part 8 — Two numerical habits worth having early

**An integer array will not accept a fractional result in place.** `a / 2` on an integer array
is fine and gives you floats, because it makes a new array. `a /= 2` refuses, because the answer
does not fit back into the integers it was asked to overwrite. Convert once, early, with
`.astype(float)` — as part 6 did before computing a cosine.

In [ ]:
ints = np.array([[1, 0, 2], [0, 3, 0]])

print("ints / 2 makes a new float array:\n", ints / 2)

try:
    ints /= 2
except TypeError as e:
    print()
    print("ints /= 2 raises:", str(e)[:70], "...")
# Verified output:
#   ints / 2 makes a new float array:
#    [[0.5 0.  1. ]
#    [0.  1.5 0. ]]
#
#   ints /= 2 raises: Cannot cast ufunc 'divide' output from dtype('float64') to dtype('int6 ...

**A logarithm of zero is not a number.** TF-IDF and every probability in Unit II take logarithms,
and a zero count is common. NumPy returns `-inf` and warns rather than raising, so the bad value
travels until something downstream turns it into `nan`.

In [ ]:
counts = np.array([3, 0, 1])

with np.errstate(divide="ignore"):      # we know what is coming; do not warn
    logs = np.log(counts)

print("logs:      ", logs)
print("any -inf?  ", np.isinf(logs).any())
print("their mean:", logs.mean(), " <- one bad value took the whole average")
print()
print("log1p adds 1 first:", np.round(np.log1p(counts), 3))
# Verified output:
#   logs:       [1.09861229       -inf 0.        ]
#   any -inf?   True
#   their mean: -inf  <- one bad value took the whole average
#
#   log1p adds 1 first: [1.386 0.    0.693]

The usual fixes are to add one before taking the logarithm — which is what smoothing does in
session 13, and what `np.log1p` does in one call — or to compute only where the count is
positive. Whichever you choose, choose deliberately: an `-inf` that reaches a mean turns the
mean into `nan`, and a `nan` in a loss function stops a model learning with no error raised
anywhere.

---
## What to remember

| you want | the call |
| --- | --- |
| what am I holding | `a.shape`, `a.dtype` — check these first |
| a new array | `np.zeros`, `np.ones`, `np.arange`, `np.linspace`, `np.eye` |
| reproducible randomness | `rng = np.random.default_rng(seed=0)` |
| one element / row / column | `a[1, 2]`, `a[1]`, `a[:, 2]` |
| an independent array | `a[1:4].copy()` — a slice alone is a view |
| filtering | `a[a > 2]`, and `&` / `|` with brackets |
| collapse an axis | `a.sum(axis=0)` — the axis you name disappears |
| keep the shape while collapsing | `a.sum(axis=1, keepdims=True)` |
| a dot product or matrix product | `a @ b` |
| a vector's length | `np.linalg.norm(v)`, or `norm(m, axis=1, keepdims=True)` |
| ranking | `np.argsort(scores)[::-1]`, `np.argmax(scores)` |
| a different shape | `a.reshape(-1, 2)`, `a[:, np.newaxis]`, `a.T` |

And the three things that will actually bite you: a slice is a view, so write to a `.copy()`;
`axis=` names the axis that disappears, so check the resulting shape rather than reasoning about
rows; and integer arrays and `log(0)` produce wrong answers quietly rather than raising.

**Next:** lab 4 builds a term-document matrix for real, and lab 7 computes the similarity matrix
of part 6 over a corpus rather than four toy documents.